# Stage 3 — Step Parser
Uses **Llama 3 (via Groq)** to label the **operation** between two consecutive steps.

**Input:** two consecutive step strings  
**Output:** operation label e.g. `Factorization`, `Apply Zero Product Rule`, `Solve Linear Equation`

Prototype here, then copy `parse_operation()` into `src/pipeline/step_parser.py`.

In [1]:
%pip install groq python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import time
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from .env
client = Groq()

# Get your free API key at: https://console.groq.com
# Add to .env file: GROQ_API_KEY=your_key_here

## Define allowed operation labels

In [3]:
ALLOWED_OPERATIONS = [
    "Factorization",
    "Expand",
    "Apply Zero Product Rule",
    "Solve Linear Equation",
    "Apply Quadratic Formula",
    "Complete the Square",
    "Simplify",
    "Collect Like Terms",
    "Move Terms",
    "Calculate Discriminant",
    "Simplify Radical",
    "Other"
]

ops_str = "\n".join(f"- {op}" for op in ALLOWED_OPERATIONS)
print(ops_str)

- Factorization
- Expand
- Apply Zero Product Rule
- Solve Linear Equation
- Apply Quadratic Formula
- Complete the Square
- Simplify
- Collect Like Terms
- Move Terms
- Calculate Discriminant
- Simplify Radical
- Other


## Build the prompt

In [35]:
SYSTEM_PROMPT = f""""Identify the operation from Step 1 → Step 2.

Options:
Factorization, Expand, Zero Product, Solve Linear, Quadratic Formula, Complete Square, Simplify, Collect like Terms, Move Terms,Calculate Discriminant, Simplify Radicals,Simplify Radicals, Other

Step 1: {{prev}}
Step 2: {{curr}}

Output JSON:
{{"operation":"<option>","confidence":0-1}}"""


def parse_operation(step_prev: str, step_curr: str, retries: int = 3) -> dict:
    """
    Use Llama 3 via Groq to identify the operation between two steps.
    Returns: {"operation": str, "confidence": float}
    """
    user_msg = f"""Step before: {step_prev}
Step after:  {step_curr}

What operation was applied?"""

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg}
                ],
                temperature=0,
                response_format={"type": "json_object"}
            )
            raw = response.choices[0].message.content.strip()
            # Strip markdown fences if model adds them
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
                raw = raw.strip()
            return json.loads(raw)

        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f"  Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  All retries failed: {e}")
                return {"operation": "Other", "confidence": 0.0}

## Test on individual step pairs

In [29]:
# Test 1: Factorization
result = parse_operation("x^2 - 5x + 6 = 0", "(x - 2)(x - 3) = 0")
print("Test 1 (expect Factorization):", result)

Test 1 (expect Factorization): {'operation': 'Factorization', 'confidence': 1}


In [30]:
# Test 2: Zero Product Rule
result = parse_operation("(x - 2)(x - 3) = 0", "x - 2 = 0 OR x - 3 = 0")
print("Test 2 (expect Apply Zero Product Rule):", result)

Test 2 (expect Apply Zero Product Rule): {'operation': 'Zero Product', 'confidence': 1}


In [31]:
# Test 3: Solve Linear
result = parse_operation("x - 2 = 0", "x = 2")
print("Test 3 (expect Solve Linear Equation):  ", result)

Test 3 (expect Solve Linear Equation):   {'operation': 'Solve Linear', 'confidence': 1}


In [32]:
# Test 4: Quadratic Formula
result = parse_operation("x^2 - 5x + 6 = 0", "x = (5 ± sqrt(25 - 24)) / 2")
print("Test 4 (expect Apply Quadratic Formula):", result)

Test 4 (expect Apply Quadratic Formula): {'operation': 'Quadratic Formula', 'confidence': 1}


In [33]:
# Test 5: Discriminant step (common in your dataset)
result = parse_operation("x = (-(3) ± √((3)² - 4(1)(-40)))/(2*1)", "Discriminant = 169")
print("Test 5 (expect Calculate Discriminant): ", result)

Test 5 (expect Calculate Discriminant):  {'operation': 'Calculate Discriminant', 'confidence': 1}


In [36]:
# Test 6: Radical simplification error (common in your dataset)
result = parse_operation("x = (5 ± √(25 - 24)) / 2", "√12 = 12")
print("Test 6 (expect Simplify Radical):       ", result)

Test 6 (expect Simplify Radical):        {'operation': 'Other', 'confidence': 0}


## Run on dataset with batching

In [23]:
import pandas as pd

DATASET_PATH = r"C:\mariam\uni\bachelor\algebra-error-detector\notebooks\quadratic_dataset.json"
BATCH_SIZE = 10
DELAY_BETWEEN_BATCHES = 3  # seconds — Groq free tier is generous but don't hammer it

with open(DATASET_PATH) as f:
    dataset = json.load(f)

# Filter out placeholder steps
PLACEHOLDER_TERMS = ['incorrect', 'placeholder', 'wrong step']
usable = [
    e for e in dataset
    if not any(
        any(p in s.lower() for p in PLACEHOLDER_TERMS)
        for s in e['steps']
    )
]
print(f"Usable entries: {len(usable)} / {len(dataset)}")

Usable entries: 2100 / 2100


In [24]:
rows = []
for i, entry in enumerate(usable[:50]):  # increase limit as needed
    steps = entry["steps"]
    for j in range(1, len(steps)):
        result = parse_operation(steps[j-1], steps[j])
        rows.append({
            "equation":   entry["equation"],
            "step_prev":  steps[j-1],
            "step_curr":  steps[j],
            "operation":  result.get("operation"),
            "confidence": result.get("confidence")
        })

    if (i + 1) % BATCH_SIZE == 0:
        print(f"  {i+1}/{len(usable)} entries processed, pausing {DELAY_BETWEEN_BATCHES}s...")
        time.sleep(DELAY_BETWEEN_BATCHES)

df = pd.DataFrame(rows)
df

  10/2100 entries processed, pausing 3s...
  20/2100 entries processed, pausing 3s...
  30/2100 entries processed, pausing 3s...
  40/2100 entries processed, pausing 3s...
  50/2100 entries processed, pausing 3s...


,equation,step_prev,step_curr,operation,confidence
0,1x^2 + -6x + 0 = 0,1x^2 + -6x + 0 = 0,(x - 6)(x - 0) = 0,Factorization,1.0
1,1x^2 + -6x + 0 = 0,(x - 6)(x - 0) = 0,x - 6 = 0 OR x - 0 = 0,Apply Zero Product,1.0
2,1x^2 + -6x + 0 = 0,x - 6 = 0 OR x - 0 = 0,x = 6 OR x = 0,Solve Linear,1.0
3,1x^2 + -6x + 0 = 0,1x^2 + -6x + 0 = 0,x = (-(-6) ± √((-6)^2 - 4(1)(0)))/(2*1),Quadratic Formula,1.0
4,1x^2 + -6x + 0 = 0,x = (-(-6) ± √((-6)^2 - 4(1)(0)))/(2*1),x = (-(-6) ± √36)/(2*1),Simplify,1.0
...,...,...,...,...,...
89,3x^2 + 0x + -12 = 0,x = (-(0) ± √((0)^2 - 4(3)(-12)))/(2*3),√12 = 12,Simplify,0.8
90,3x^2 + 0x + -12 = 0,3x^2 + 0x + -12 = 0,(x + 1)(x + 4) = 0,Factorization,1.0
91,1x^2 + 12x + 20 = 0,1x^2 + 12x + 20 = 0,(x - -2)(x - -10) = 0,Factorization,1.0
92,1x^2 + 12x + 20 = 0,(x - -2)(x - -10) = 0,x - -2 = 0 OR x - -10 = 0,Apply Zero Product,1.0


In [25]:
# Operation distribution
print(df["operation"].value_counts())

# Flag low confidence for review
print(f"\nLow confidence (<0.8): {len(df[df['confidence'] < 0.8])} rows")
df[df["confidence"] < 0.8]

operation
Factorization         29
Quadratic Formula     21
Simplify              18
Apply Zero Product    14
Solve Linear           9
Simplify Radicals      2
Other                  1
Name: count, dtype: int64

Low confidence (<0.8): 0 rows


,equation,step_prev,step_curr,operation,confidence


## ✅ Once prompt is good → copy `parse_operation()` to `src/pipeline/step_parser.py`